# Trendyol 2026 category-aware PU reranker

This notebook turns the previous leaderboard probes into a reproducible experiment and adds the most useful next feature family from the analysis: query-to-category compatibility.

Known public probes from 2026-06-28:

| file | idea | public score |
| --- | --- | ---: |
| `submission_lexical_t575.csv` | lexical threshold 5.75 | 0.74 |
| `submission_lexical_t700.csv` | lexical threshold 7.00 | 0.61 |
| `submission_lexical_t450.csv` | lexical threshold 4.50 | 0.76 |
| `submission_lexical_top20.csv` | fixed top 20 per query | 0.54 |
| `submission_pu_lgbm_v1_r20.csv` | PU LightGBM, 20 percent positive rate | 0.78 |

The main change here is to train a lightweight category predictor from positive training pairs, then add category match/rank/probability features to the PU LightGBM reranker. Set `FULL_RUN = True` for a full leaderboard candidate. The default is a development run that trains on smaller samples and avoids writing a full 3.36M-row score file.


In [1]:
import importlib.util
import subprocess
import sys

required_packages = {
    "rapidfuzz": "rapidfuzz",
    "text-unidecode": "text_unidecode",
}
missing = [
    package
    for package, module in required_packages.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("required packages already available")


required packages already available


In [2]:
from __future__ import annotations

import csv
import gc
import math
import os
import subprocess
import sys
import time
import warnings
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    display
except NameError:
    def display(obj):
        print(obj)

REQUIRED_DATA_FILES = {
    "items.csv",
    "terms.csv",
    "training_pairs.csv",
    "submission_pairs.csv",
    "sample_submission.csv",
}
KAGGLE_INPUT_DIR = Path("/kaggle/input")
IS_KAGGLE = Path("/kaggle").exists() or bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))


def unique_paths(paths):
    seen: set[Path] = set()
    for path in paths:
        resolved = path.resolve()
        if resolved in seen:
            continue
        seen.add(resolved)
        yield resolved


def existing_dirs(paths):
    yield from (path for path in unique_paths(paths) if path.is_dir())


def local_data_dirs() -> tuple[Path, ...]:
    cwd = Path.cwd()
    return (cwd / "data", cwd, cwd.parent / "data", cwd.parent)


def kaggle_data_dirs():
    if not (IS_KAGGLE and KAGGLE_INPUT_DIR.is_dir()):
        return
    yield KAGGLE_INPUT_DIR
    yield from (path for path in KAGGLE_INPUT_DIR.iterdir() if path.is_dir())
    yield from (path for path in KAGGLE_INPUT_DIR.glob("*/*") if path.is_dir())
    yield from (path.parent for path in KAGGLE_INPUT_DIR.rglob("items.csv"))


def data_dir_candidates():
    yield from local_data_dirs()
    yield from kaggle_data_dirs()


def has_required_data(path: Path) -> bool:
    return path.is_dir() and all((path / name).is_file() for name in REQUIRED_DATA_FILES)


def find_data_dir() -> Path:
    for path in unique_paths(data_dir_candidates()):
        if has_required_data(path):
            return path

    searched = ", ".join(str(path) for path in local_data_dirs())
    if IS_KAGGLE:
        searched += ", /kaggle/input/**"
    raise FileNotFoundError(f"Could not find the competition CSV files under {searched}")


def find_first_file(file_name: str, roots: list[Path]) -> Path | None:
    for root in existing_dirs(roots):
        direct = root / file_name
        if direct.is_file():
            return direct
        matches = sorted(root.rglob(file_name))
        if matches:
            return matches[0]
    return None


def source_dir_candidates(data_dir: Path):
    root = data_dir.parent if data_dir.name == "data" else data_dir
    yield from (Path.cwd() / "src", Path.cwd().parent / "src", root / "src")
    if IS_KAGGLE and KAGGLE_INPUT_DIR.is_dir():
        yield from KAGGLE_INPUT_DIR.glob("*/src")
        yield from (path.parent for path in KAGGLE_INPUT_DIR.rglob("lexical_baseline.py"))


def find_source_dir(data_dir: Path) -> Path | None:
    for path in existing_dirs(source_dir_candidates(data_dir)):
        if (path / "lexical_baseline.py").is_file():
            return path
    return None


def detect_cuda_gpus() -> list[dict[str, object]]:
    command = ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]
    try:
        result = subprocess.run(command, capture_output=True, text=True, timeout=10, check=False)
    except (FileNotFoundError, subprocess.SubprocessError):
        result = None

    if result is not None and result.returncode == 0:
        names = [line.strip() for line in result.stdout.splitlines() if line.strip()]
        if names:
            return [{"id": idx, "name": name} for idx, name in enumerate(names)]

    visible = os.environ.get("CUDA_VISIBLE_DEVICES", "")
    if visible and visible not in {"-1", "NoDevFiles"}:
        devices = [part.strip() for part in visible.split(",") if part.strip()]
        return [{"id": idx, "name": f"CUDA device {device}"} for idx, device in enumerate(devices)]
    return []


DATA_DIR = find_data_dir()
ROOT = DATA_DIR.parent if DATA_DIR.name == "data" else DATA_DIR
OUT_DIR = Path("/kaggle/working/outputs") if IS_KAGGLE else ROOT / "outputs"
NB_OUT_DIR = OUT_DIR / "category_aware_notebook"
NB_OUT_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = [OUT_DIR, ROOT / "outputs", ROOT, DATA_DIR]
if IS_KAGGLE:
    SEARCH_ROOTS.append(KAGGLE_INPUT_DIR)
LEXICAL_SCORES_PATH = find_first_file("lexical_scores.csv", SEARCH_ROOTS) or OUT_DIR / "lexical_scores.csv"
OLD_PU_SCORES_PATH = find_first_file("pu_lgbm_v1_scores.csv", SEARCH_ROOTS)

SRC_DIR = find_source_dir(DATA_DIR)
if SRC_DIR is None:
    raise FileNotFoundError("Could not find src/lexical_baseline.py. On Kaggle, attach this repo/source folder as a Dataset too.")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from lexical_baseline import load_items, load_terms, normalize_text, pair_features

GPU_DEVICES = detect_cuda_gpus()
GPU_IDS = [int(device["id"]) for device in GPU_DEVICES]
USE_GPU = bool(GPU_IDS) and os.environ.get("USE_LGBM_GPU", "1") != "0"
USE_DUAL_T4 = USE_GPU and len(GPU_IDS) >= 2 and os.environ.get("USE_DUAL_T4", "1") != "0"

SEED = 42
rng = np.random.default_rng(SEED)

FULL_RUN = True
NEGATIVE_SAMPLE_SCALE = 1.0 if FULL_RUN else 0.08
N_POS = 0 if FULL_RUN else 50_000
CATEGORY_MAX_EXAMPLES = None if FULL_RUN else 120_000
CATEGORY_MAX_FEATURES = 90_000 if FULL_RUN else 45_000
CATEGORY_MAX_ITER = 25 if FULL_RUN else 10
PREDICT_FULL_SUBMISSION = FULL_RUN

SETUP_SUMMARY = {
    "kaggle": IS_KAGGLE,
    "root": ROOT,
    "data_dir": DATA_DIR,
    "src_dir": SRC_DIR,
    "full_run": FULL_RUN,
    "notebook_outputs": NB_OUT_DIR,
    "lexical_scores": f"{LEXICAL_SCORES_PATH} exists={LEXICAL_SCORES_PATH.exists()}",
    "old_pu_scores": OLD_PU_SCORES_PATH,
    "gpu_devices": GPU_DEVICES if GPU_DEVICES else "none",
    "lightgbm_gpu": f"{USE_GPU} dual_t4={USE_DUAL_T4}",
}
for key, value in SETUP_SUMMARY.items():
    print(f"{key}={value}")


kaggle=False
root=/Users/z3lka/trendyol_e_ticaret_hack
data_dir=/Users/z3lka/trendyol_e_ticaret_hack/data
src_dir=/Users/z3lka/trendyol_e_ticaret_hack/src
full_run=True
notebook_outputs=/Users/z3lka/trendyol_e_ticaret_hack/outputs/category_aware_notebook
lexical_scores=/Users/z3lka/trendyol_e_ticaret_hack/outputs/lexical_scores.csv exists=True
old_pu_scores=/Users/z3lka/trendyol_e_ticaret_hack/outputs/pu_lgbm_v1_scores.csv
gpu_devices=none
lightgbm_gpu=False dual_t4=False


## Data audit

The competition data is positive-only: every row in `training_pairs.csv` is label 1. The PU model therefore samples likely negatives from the unlabeled submission candidate pool and treats them as noisy negatives.


In [3]:
def count_csv_rows(path: Path) -> int:
    with path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.reader(handle)
        next(reader, None)
        return sum(1 for _ in reader)

files = [
    DATA_DIR / "items.csv",
    DATA_DIR / "terms.csv",
    DATA_DIR / "training_pairs.csv",
    DATA_DIR / "submission_pairs.csv",
    DATA_DIR / "sample_submission.csv",
]

audit = pd.DataFrame(
    {
        "file": [path.name for path in files],
        "rows": [count_csv_rows(path) for path in files],
        "size_mb": [path.stat().st_size / 1024 / 1024 for path in files],
    }
)
display(audit)

train_label_counts = pd.read_csv(DATA_DIR / "training_pairs.csv", usecols=["label"])["label"].value_counts(dropna=False)
print(train_label_counts.to_string())


,file,rows,size_mb
0,items.csv,962873,383.431083
1,terms.csv,50153,1.550133
2,training_pairs.csv,250000,11.682534
3,submission_pairs.csv,3359679,163.406036
4,sample_submission.csv,3359679,67.284844


label
1    250000


In [4]:
terms_df = pd.read_csv(DATA_DIR / "terms.csv")
items_meta = pd.read_csv(DATA_DIR / "items.csv", usecols=["item_id", "category"])
train_pairs_all = pd.read_csv(DATA_DIR / "training_pairs.csv", usecols=["term_id", "item_id"])

print(f"terms={len(terms_df):,}")
print(f"items={len(items_meta):,}")
print(f"positive pairs={len(train_pairs_all):,}")
print(f"unique full categories in items={items_meta['category'].nunique():,}")


terms=50,153
items=962,873
positive pairs=250,000
unique full categories in items=2,932


## Base lexical feature set

These are the exact feature names used by the previous best local artifact, `outputs/pu_lgbm_v1.joblib`.


In [5]:
BASE_FEATURE_NAMES = [
    "n_query_tokens",
    "title_cov",
    "category_cov",
    "brand_cov",
    "attr_cov",
    "full_cov",
    "title_ratio",
    "category_ratio",
    "brand_ratio",
    "query_in_title",
    "query_in_category",
    "query_in_brand",
    "full_complete",
    "full_high",
    "title_complete",
    "category_complete",
    "brand_complete",
    "short_complete",
    "gender_mismatch",
    "age_mismatch",
    "color_mismatch",
    "weak_match",
    "title_len",
    "category_len",
    "brand_len",
    "attr_len",
    "lexical_score",
]

previous_scores = pd.DataFrame(
    [
        ("lexical_t575", 0.74, 0.175),
        ("lexical_t700", 0.61, 0.110),
        ("lexical_t450", 0.76, 0.203),
        ("lexical_top20", 0.54, np.nan),
        ("pu_lgbm_v1_r20", 0.78, 0.200),
    ],
    columns=["submission", "public_score", "positive_rate"],
)
display(previous_scores)


,submission,public_score,positive_rate
0,lexical_t575,0.74,0.175
1,lexical_t700,0.61,0.110
2,lexical_t450,0.76,0.203
3,lexical_top20,0.54,NaN
4,pu_lgbm_v1_r20,0.78,0.200


## Category compatibility model

The training positives tell us which item categories are plausible for a query. This section trains query-to-category classifiers for level 1 and level 2 category paths. Level 2 has enough detail to be useful while staying small enough for a fast linear model.


In [6]:
CATEGORY_LEVELS = ("cat_l1", "cat_l2")
CATEGORY_TOPK = 5


def split_category(value: object) -> tuple[str, str, str, str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        parts = ["unknown"]
    else:
        parts = [normalize_text(part) for part in str(value).split("/")]
        parts = [part for part in parts if part]
        if not parts:
            parts = ["unknown"]

    cat_l1 = parts[0]
    cat_l2 = "/".join(parts[:2]) if len(parts) >= 2 else cat_l1
    cat_l3 = "/".join(parts[:3]) if len(parts) >= 3 else cat_l2
    cat_leaf = "/".join(parts)
    return cat_l1, cat_l2, cat_l3, cat_leaf


def add_category_levels(frame: pd.DataFrame, category_col: str = "category") -> pd.DataFrame:
    levels = pd.DataFrame(
        frame[category_col].map(split_category).tolist(),
        columns=["cat_l1", "cat_l2", "cat_l3", "cat_leaf"],
        index=frame.index,
    )
    return pd.concat([frame, levels], axis=1)


def build_category_examples(
    terms: pd.DataFrame,
    items: pd.DataFrame,
    train_pairs: pd.DataFrame,
    max_examples: int | None = None,
    seed: int = SEED,
) -> pd.DataFrame:
    examples = (
        train_pairs.merge(terms, on="term_id", how="left")
        .merge(items, on="item_id", how="left")
        .dropna(subset=["query", "category"])
    )
    examples = add_category_levels(examples)
    examples["query_norm"] = examples["query"].map(normalize_text)

    # Keep one row per query/category path. This prevents a query with many positive
    # items in the same category from dominating the classifier.
    examples = examples.drop_duplicates(["query_norm", "cat_l2"])
    if max_examples is not None and len(examples) > max_examples:
        examples = examples.sample(n=max_examples, random_state=seed)

    cols = ["query_norm", "cat_l1", "cat_l2", "cat_l3", "cat_leaf"]
    return examples[cols].reset_index(drop=True)


category_examples = build_category_examples(
    terms_df,
    items_meta,
    train_pairs_all,
    max_examples=CATEGORY_MAX_EXAMPLES,
)
print(category_examples.shape)
for col in ["cat_l1", "cat_l2", "cat_l3", "cat_leaf"]:
    print(f"{col}: {category_examples[col].nunique():,}")

display(category_examples.head())


(25911, 5)
cat_l1: 15
cat_l2: 117
cat_l3: 906
cat_leaf: 2,264


,query_norm,cat_l1,cat_l2,cat_l3,cat_leaf
0,panoroma,ev mobilya,ev mobilya/ev,ev mobilya/ev/ev dekorasyon,ev mobilya/ev/ev dekorasyon/kirlent kirlent ki...
1,matkap ucu,bahce elektrikli el aletleri,bahce elektrikli el aletleri/elektrikli el ale...,bahce elektrikli el aletleri/elektrikli el ale...,bahce elektrikli el aletleri/elektrikli el ale...
2,soundbar,elektronik,elektronik/hoparlor projeksiyon sistemleri,elektronik/hoparlor projeksiyon sistemleri/sou...,elektronik/hoparlor projeksiyon sistemleri/sou...
3,granul kahve,supermarket,supermarket/gida icecek,supermarket/gida icecek/kahve,supermarket/gida icecek/kahve/granul kahve
4,oyuncu dizustu bilgisayari,elektronik,elektronik/bilgisayar grubu,elektronik/bilgisayar grubu/bilgisayarlar,elektronik/bilgisayar grubu/bilgisayarlar/dizu...


In [7]:
class CategorySignal:
    def __init__(
        self,
        levels: tuple[str, ...] = CATEGORY_LEVELS,
        topk: int = CATEGORY_TOPK,
        max_features: int = CATEGORY_MAX_FEATURES,
        max_iter: int = CATEGORY_MAX_ITER,
        min_class_count: int = 2,
        seed: int = SEED,
    ):
        self.levels = levels
        self.topk = topk
        self.max_features = max_features
        self.max_iter = max_iter
        self.min_class_count = min_class_count
        self.seed = seed
        self.vectorizer: TfidfVectorizer | None = None
        self.models: dict[str, SGDClassifier] = {}

    def fit(self, examples: pd.DataFrame) -> "CategorySignal":
        self.vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(2, 5),
            min_df=2,
            max_features=self.max_features,
            dtype=np.float32,
            sublinear_tf=True,
        )
        x_all = self.vectorizer.fit_transform(examples["query_norm"])
        print(f"category tfidf shape={x_all.shape}")

        for level in self.levels:
            counts = examples[level].value_counts()
            keep_labels = counts.index[counts >= self.min_class_count]
            keep = examples[level].isin(keep_labels).to_numpy()
            y = examples.loc[keep, level].to_numpy()
            x = x_all[keep]
            clf = SGDClassifier(
                loss="log_loss",
                penalty="l2",
                alpha=2e-5,
                max_iter=self.max_iter,
                tol=1e-3,
                class_weight="balanced",
                random_state=self.seed,
                n_jobs=-1,
            )
            start = time.time()
            clf.fit(x, y)
            self.models[level] = clf
            elapsed = time.time() - start
            print(f"{level}: classes={len(clf.classes_):,} examples={len(y):,} seconds={elapsed:.1f}")
        return self

    def predict_terms(self, terms: pd.DataFrame, chunk_size: int = 25_000) -> pd.DataFrame:
        if self.vectorizer is None:
            raise RuntimeError("fit the category signal first")

        term_ids = terms["term_id"].to_numpy()
        queries = terms["query"].map(normalize_text).to_numpy()
        chunks: list[pd.DataFrame] = []

        for start in tqdm(range(0, len(terms), chunk_size), desc="category topk"):
            end = min(start + chunk_size, len(terms))
            x = self.vectorizer.transform(queries[start:end])
            out: dict[str, object] = {"term_id": term_ids[start:end]}

            for level, clf in self.models.items():
                proba = clf.predict_proba(x)
                k = min(self.topk, proba.shape[1])
                idx = np.argpartition(proba, -k, axis=1)[:, -k:]
                local_order = np.argsort(-np.take_along_axis(proba, idx, axis=1), axis=1)
                idx = np.take_along_axis(idx, local_order, axis=1)
                scores = np.take_along_axis(proba, idx, axis=1).astype(np.float32)
                labels = clf.classes_[idx]

                for rank in range(k):
                    out[f"{level}_top{rank + 1}"] = labels[:, rank]
                    out[f"{level}_prob{rank + 1}"] = scores[:, rank]

            chunks.append(pd.DataFrame(out))

        return pd.concat(chunks, ignore_index=True)


category_signal = CategorySignal().fit(category_examples)
term_category_topk = category_signal.predict_terms(terms_df)
term_category_topk_path = NB_OUT_DIR / "term_category_topk.csv"
term_category_topk.to_csv(term_category_topk_path, index=False)
print(f"wrote {term_category_topk_path}")
display(term_category_topk.head())


category tfidf shape=(25911, 34736)
cat_l1: classes=15 examples=25,911 seconds=0.1
cat_l2: classes=116 examples=25,910 seconds=0.3


category topk:   0%|          | 0/3 [00:00<?, ?it/s]

wrote /Users/z3lka/trendyol_e_ticaret_hack/outputs/category_aware_notebook/term_category_topk.csv


,term_id,cat_l1_top1,cat_l1_prob1,cat_l1_top2,cat_l1_prob2,cat_l1_top3,cat_l1_prob3,cat_l1_top4,cat_l1_prob4,cat_l1_top5,...,cat_l2_top1,cat_l2_prob1,cat_l2_top2,cat_l2_prob2,cat_l2_top3,cat_l2_prob3,cat_l2_top4,cat_l2_prob4,cat_l2_top5,cat_l2_prob5
0,TERM_f2b61db2,giyim,0.940133,anne bebek cocuk,0.012553,supermarket,0.010211,aksesuar,0.009832,ayakkabi,...,giyim/elbise,0.434049,giyim/bebek giyim,0.119159,giyim/hamile giyim,0.056849,giyim/abiye mezuniyet elbisesi,0.048582,giyim/salopet tulum,0.047973
1,TERM_00247623,anne bebek cocuk,0.576763,otomobil motosiklet,0.207185,ev mobilya,0.119443,supermarket,0.032942,giyim,...,anne bebek cocuk/cocuk gerecleri,0.542781,otomobil motosiklet/otomobil,0.055155,ev mobilya/mobilya,0.046321,supermarket/anne ve bebek bakim,0.035372,otomobil motosiklet/otomobil yedek parca,0.029268
2,TERM_c977024b,otomobil motosiklet,0.415113,ev mobilya,0.177906,spor outdoor,0.130215,bahce elektrikli el aletleri,0.076678,supermarket,...,otomobil motosiklet/lastik jant,0.620983,ev mobilya/ev,0.032754,bahce elektrikli el aletleri/elektrikli el ale...,0.020556,otomobil motosiklet/otomobil yedek parca,0.019849,supermarket/ev tuketim malzemesi,0.017496
3,TERM_dbdbddbb,giyim,0.959114,ayakkabi,0.012210,aksesuar,0.006514,spor outdoor,0.003065,ev mobilya,...,giyim/buyuk beden,0.319718,giyim/tesettur giyim,0.296202,giyim/ust giyim,0.174410,giyim/takimlar,0.022385,giyim/dis giyim,0.007293
4,TERM_c06329e3,ayakkabi,0.523751,giyim,0.252475,aksesuar,0.116709,anne bebek cocuk,0.047358,kitap,...,ayakkabi/spor ayakkabi,0.107956,giyim/ic giyim,0.076732,giyim/salopet tulum,0.061447,giyim/ev giyim,0.055017,giyim/takimlar,0.042131


## Feature builders

The new features answer questions like: Is this item's level 2 category the model's top category for the query? Is it in the top 3 or top 5? What probability did the category model assign to that category?


In [8]:
def build_term_category_lookup(topk: pd.DataFrame, levels: tuple[str, ...] = CATEGORY_LEVELS, k: int = CATEGORY_TOPK) -> dict[str, dict[str, object]]:
    lookup: dict[str, dict[str, object]] = {}
    for row in topk.itertuples(index=False):
        row_dict = row._asdict()
        entry: dict[str, object] = {}
        for level in levels:
            scores: dict[str, float] = {}
            for rank in range(1, k + 1):
                label = row_dict.get(f"{level}_top{rank}")
                score = row_dict.get(f"{level}_prob{rank}")
                if isinstance(label, str) and label:
                    scores[label] = float(score)
            entry[level] = scores
            entry[f"{level}_top1"] = next(iter(scores), "")
            entry[f"{level}_top1_prob"] = next(iter(scores.values()), 0.0)
        lookup[row_dict["term_id"]] = entry
    return lookup


def build_item_category_lookup(items: pd.DataFrame) -> dict[str, dict[str, str]]:
    levels = add_category_levels(items[["item_id", "category"]].copy())
    return {
        row.item_id: {
            "cat_l1": row.cat_l1,
            "cat_l2": row.cat_l2,
            "cat_l3": row.cat_l3,
            "cat_leaf": row.cat_leaf,
        }
        for row in levels.itertuples(index=False)
    }


CATEGORY_FEATURE_NAMES = []
for level in CATEGORY_LEVELS:
    CATEGORY_FEATURE_NAMES.extend(
        [
            f"{level}_top1_match",
            f"{level}_top3_match",
            f"{level}_top5_match",
            f"{level}_candidate_prob",
            f"{level}_top1_prob",
            f"{level}_rank",
        ]
    )

FEATURE_NAMES = BASE_FEATURE_NAMES + CATEGORY_FEATURE_NAMES


def category_pair_features(
    term_id: str,
    item_id: str,
    term_category_lookup: dict[str, dict[str, object]],
    item_category_lookup: dict[str, dict[str, str]],
) -> dict[str, float]:
    term_entry = term_category_lookup.get(term_id, {})
    item_entry = item_category_lookup.get(item_id, {})
    out: dict[str, float] = {}

    for level in CATEGORY_LEVELS:
        scores = term_entry.get(level, {})
        if not isinstance(scores, dict):
            scores = {}
        ordered_labels = list(scores.keys())
        candidate = item_entry.get(level, "")
        rank = ordered_labels.index(candidate) + 1 if candidate in scores else 0

        out[f"{level}_top1_match"] = float(rank == 1)
        out[f"{level}_top3_match"] = float(1 <= rank <= 3)
        out[f"{level}_top5_match"] = float(1 <= rank <= 5)
        out[f"{level}_candidate_prob"] = float(scores.get(candidate, 0.0))
        out[f"{level}_top1_prob"] = float(term_entry.get(f"{level}_top1_prob", 0.0))
        out[f"{level}_rank"] = float(rank)

    return out


term_category_lookup = build_term_category_lookup(term_category_topk)
item_category_lookup = build_item_category_lookup(items_meta)
print(f"feature count={len(FEATURE_NAMES)} base={len(BASE_FEATURE_NAMES)} category={len(CATEGORY_FEATURE_NAMES)}")


feature count=39 base=27 category=12


In [9]:
def rows_to_matrix(
    rows: pd.DataFrame,
    terms: dict[str, dict[str, object]],
    items: dict[str, tuple[str, str, str, str, str, str]],
    term_category_lookup: dict[str, dict[str, object]],
    item_category_lookup: dict[str, dict[str, str]],
    desc: str,
) -> np.ndarray:
    matrix = np.empty((len(rows), len(FEATURE_NAMES)), dtype=np.float32)
    iterator = tqdm(rows.itertuples(index=False), total=len(rows), desc=desc, leave=False)
    for i, row in enumerate(iterator):
        features = pair_features(terms[row.term_id], items[row.item_id])
        features.update(category_pair_features(row.term_id, row.item_id, term_category_lookup, item_category_lookup))
        matrix[i] = [features.get(name, 0.0) for name in FEATURE_NAMES]
    return matrix


def sample_frame(frame: pd.DataFrame, mask: pd.Series, n: int, seed: int) -> pd.DataFrame:
    subset = frame.loc[mask]
    if len(subset) <= n:
        return subset
    return subset.sample(n=n, random_state=seed)


def build_unlabeled_negatives(
    lexical_scores_path: Path = OUT_DIR / "lexical_scores.csv",
    scale: float = NEGATIVE_SAMPLE_SCALE,
    seed: int = SEED,
    low_max: float = 2.0,
    mid_max: float = 4.5,
    hard_max: float = 6.5,
) -> pd.DataFrame:
    pairs = pd.read_csv(DATA_DIR / "submission_pairs.csv", usecols=["id", "term_id", "item_id"])
    scores = pd.read_csv(lexical_scores_path, usecols=["score"])
    if len(pairs) != len(scores):
        raise ValueError("submission_pairs and lexical_scores row counts differ")

    pairs["score"] = scores["score"].astype("float32").to_numpy()
    n_low = int(350_000 * scale)
    n_mid = int(350_000 * scale)
    n_hard = int(150_000 * scale)

    parts = [
        sample_frame(pairs, pairs["score"] < low_max, n_low, seed),
        sample_frame(pairs, (pairs["score"] >= low_max) & (pairs["score"] < mid_max), n_mid, seed + 1),
        sample_frame(pairs, (pairs["score"] >= mid_max) & (pairs["score"] < hard_max), n_hard, seed + 2),
    ]
    negatives = pd.concat(parts, ignore_index=True).drop_duplicates("id")
    print("negative score distribution")
    print(negatives["score"].describe(percentiles=[0.1, 0.5, 0.9, 0.99]).to_string())
    return negatives[["term_id", "item_id"]]


## Train the category-aware PU LightGBM

This mirrors the previous `pu_lgbm_v1` setup, but the matrix now includes category compatibility features. The validation score is synthetic because the negative class is sampled from unlabeled data, so leaderboard validation still requires a Kaggle submission after the daily quota resets.


In [10]:
terms = load_terms(DATA_DIR / "terms.csv")
items = load_items(DATA_DIR / "items.csv")

positives = train_pairs_all.copy()
if N_POS and len(positives) > N_POS:
    positives = positives.sample(n=N_POS, random_state=SEED)

negatives = build_unlabeled_negatives(scale=NEGATIVE_SAMPLE_SCALE)

x_pos = rows_to_matrix(positives, terms, items, term_category_lookup, item_category_lookup, "positive features")
x_neg = rows_to_matrix(negatives, terms, items, term_category_lookup, item_category_lookup, "negative features")
y_pos = np.ones(len(x_pos), dtype=np.int8)
y_neg = np.zeros(len(x_neg), dtype=np.int8)

x = np.vstack([x_pos, x_neg])
y = np.concatenate([y_pos, y_neg])
print(f"matrix={x.shape} positives={y.sum():,} positive_rate={y.mean():.4f}")

del x_pos, x_neg, y_pos, y_neg
gc.collect()


items: 962873it [00:23, 40193.74it/s]


negative score distribution
count    850000.000000
mean          2.166150
std           2.279237
min          -2.707246
10%          -0.488258
50%           2.457500
90%           5.760868
99%           6.378401
max           6.499945


positive features:   0%|          | 0/250000 [00:00<?, ?it/s]

negative features:   0%|          | 0/850000 [00:00<?, ?it/s]

matrix=(1100000, 39) positives=250,000 positive_rate=0.2273


62

In [15]:
x_train, x_valid, y_train, y_valid = train_test_split(
    x,
    y,
    test_size=0.15,
    random_state=SEED,
    stratify=y,
)

clf = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=10000 if FULL_RUN else 350,
    learning_rate=0.035,
    num_leaves=128,
    min_child_samples=80,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=2.0,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)

clf.fit(
    x_train,
    y_train,
    eval_set=[(x_valid, y_valid)],
    eval_metric="binary_logloss",
    callbacks=[lgb.log_evaluation(period=50)],
)

valid_prob = clf.predict_proba(x_valid)[:, 1]
best_score = -1.0
best_threshold = 0.5
for threshold in np.linspace(0.05, 0.95, 91):
    pred = (valid_prob >= threshold).astype(np.int8)
    score = f1_score(y_valid, pred, average="macro")
    if score > best_score:
        best_score = float(score)
        best_threshold = float(threshold)

print(f"synthetic valid macro_f1={best_score:.5f} threshold={best_threshold:.3f}")
importances = pd.Series(clf.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=False)
display(importances.head(35).to_frame("importance"))

model_path = NB_OUT_DIR / "category_aware_pu_lgbm.joblib"
joblib.dump(
    {
        "model": clf,
        "features": FEATURE_NAMES,
        "threshold": best_threshold,
        "category_levels": CATEGORY_LEVELS,
        "category_topk_path": str(term_category_topk_path),
        "full_run": FULL_RUN,
    },
    model_path,
)
print(f"wrote {model_path}")


[LightGBM] [Info] Number of positive: 212500, number of negative: 722500
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012687 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2283
[LightGBM] [Info] Number of data points in the train set: 935000, number of used features: 39
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[50]	valid_0's binary_logloss: 0.325034
[100]	valid_0's binary_logloss: 0.267671
[150]	valid_0's binary_logloss: 0.248396
[200]	valid_0's binary_logloss: 0.237757
[250]	valid_0's binary_logloss: 0.228953
[300]	valid_0's binary_logloss: 0.222191
[350]	valid_0's binary_logloss: 0.214353
[400]	valid_0's binary_logloss: 0.207175
[450]	valid_0's binary_logloss: 0.200746
[500]	valid_0's binary_logloss: 0.195154
[550]	valid_0's binary_logloss

/Users/z3lka/kaggles6e6/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


synthetic valid macro_f1=0.98504 threshold=0.450


,importance
category_ratio,170470
cat_l2_top1_prob,158442
cat_l1_candidate_prob,148855
cat_l2_candidate_prob,147629
cat_l1_top1_prob,125663
title_ratio,84086
attr_len,77351
lexical_score,75347
category_len,62099
brand_ratio,47868


wrote /Users/z3lka/trendyol_e_ticaret_hack/outputs/category_aware_notebook/category_aware_pu_lgbm.joblib


## Predict and create candidate submissions

The public best from earlier used a 20 percent positive rate. This notebook writes rate-based submissions around that point. It also supports a rank-normalized blend with the previous best PU scores, which is often more stable than replacing the previous model outright.


In [16]:
def predict_pairs_to_file(
    clf: lgb.LGBMClassifier,
    output_path: Path,
    chunk_size: int = 150_000,
    row_limit: int | None = None,
) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    written = 0

    with output_path.open("w", newline="", encoding="utf-8") as out_handle:
        writer = csv.writer(out_handle)
        writer.writerow(["id", "term_id", "prob"])

        reader = pd.read_csv(DATA_DIR / "submission_pairs.csv", chunksize=chunk_size)
        for chunk in tqdm(reader, desc="predict chunks"):
            if row_limit is not None:
                remaining = row_limit - written
                if remaining <= 0:
                    break
                chunk = chunk.head(remaining)

            matrix = rows_to_matrix(chunk, terms, items, term_category_lookup, item_category_lookup, "chunk features")
            prob = clf.predict_proba(matrix)[:, 1]
            writer.writerows(
                (row_id, term_id, f"{score:.8f}")
                for row_id, term_id, score in zip(chunk["id"], chunk["term_id"], prob)
            )
            written += len(chunk)
            del matrix, prob, chunk
            gc.collect()

    print(f"wrote {output_path} rows={written:,}")
    return output_path


score_path = NB_OUT_DIR / "category_aware_scores.csv"
if PREDICT_FULL_SUBMISSION:
    predict_pairs_to_file(clf, score_path)
else:
    preview_score_path = NB_OUT_DIR / "category_aware_scores_preview.csv"
    predict_pairs_to_file(clf, preview_score_path, chunk_size=25_000, row_limit=25_000)
    display(pd.read_csv(preview_score_path).head())


predict chunks: 0it [00:00, ?it/s]

chunk features:   0%|          | 0/150000 [00:00<?, ?it/s]

/Users/z3lka/kaggles6e6/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


chunk features:   0%|          | 0/150000 [00:00<?, ?it/s]

/Users/z3lka/kaggles6e6/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

chunk features:   0%|          | 0/150000 [00:00<?, ?it/s]

/Users/z3lka/kaggles6e6/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


chunk features:   0%|          | 0/59679 [00:00<?, ?it/s]

/Users/z3lka/kaggles6e6/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


wrote /Users/z3lka/trendyol_e_ticaret_hack/outputs/category_aware_notebook/category_aware_scores.csv rows=3,359,679


In [17]:
def make_rate_submission(scores_path: Path, output_path: Path, rate: float = 0.20, score_col: str = "prob") -> Path:
    scores = pd.read_csv(scores_path)
    threshold = scores[score_col].quantile(1.0 - rate)
    pred = (scores[score_col] >= threshold).astype("int8")
    submission = pd.DataFrame({"id": scores["id"], "prediction": pred})
    submission.to_csv(output_path, index=False)
    print(
        f"{output_path.name}: threshold={threshold:.8f} "
        f"positives={int(pred.sum()):,} rate={pred.mean():.5f} rows={len(pred):,}"
    )
    return output_path


if PREDICT_FULL_SUBMISSION:
    for rate in [0.18, 0.20, 0.22, 0.24]:
        make_rate_submission(
            score_path,
            NB_OUT_DIR / f"submission_category_aware_r{int(rate * 100):02d}.csv",
            rate=rate,
        )


submission_category_aware_r18.csv: threshold=0.04799829 positives=604,743 rate=0.18000 rows=3,359,679
submission_category_aware_r20.csv: threshold=0.03287616 positives=671,936 rate=0.20000 rows=3,359,679
submission_category_aware_r22.csv: threshold=0.02400655 positives=739,130 rate=0.22000 rows=3,359,679
submission_category_aware_r24.csv: threshold=0.01822454 positives=806,323 rate=0.24000 rows=3,359,679


In [18]:
def rank_normalize(series: pd.Series) -> pd.Series:
    return series.rank(method="average", pct=True).astype("float32")


def blend_scores(
    new_scores_path: Path,
    old_scores_path: Path = OUT_DIR / "pu_lgbm_v1_scores.csv",
    output_path: Path = NB_OUT_DIR / "category_aware_blend_scores.csv",
    new_weight: float = 0.65,
) -> Path:
    new_scores = pd.read_csv(new_scores_path)
    old_scores = pd.read_csv(old_scores_path)
    old_col = "prob" if "prob" in old_scores.columns else "score"
    merged = new_scores.merge(old_scores[["id", old_col]], on="id", how="inner", suffixes=("_new", "_old"))
    if len(merged) != len(new_scores):
        raise ValueError("score files do not align on id")

    new_col = "prob_new" if "prob_new" in merged.columns else "prob"
    old_merged_col = f"{old_col}_old" if f"{old_col}_old" in merged.columns else old_col
    merged["prob"] = (
        new_weight * rank_normalize(merged[new_col])
        + (1.0 - new_weight) * rank_normalize(merged[old_merged_col])
    )
    out = merged[["id", "term_id", "prob"]]
    out.to_csv(output_path, index=False)
    print(f"wrote {output_path} rows={len(out):,} new_weight={new_weight:.2f}")
    return output_path


if PREDICT_FULL_SUBMISSION and (OUT_DIR / "pu_lgbm_v1_scores.csv").exists():
    blend_path = blend_scores(score_path)
    for rate in [0.18, 0.20, 0.22, 0.24]:
        make_rate_submission(
            blend_path,
            NB_OUT_DIR / f"submission_category_aware_blend_r{int(rate * 100):02d}.csv",
            rate=rate,
        )


wrote /Users/z3lka/trendyol_e_ticaret_hack/outputs/category_aware_notebook/category_aware_blend_scores.csv rows=3,359,679 new_weight=0.65
submission_category_aware_blend_r18.csv: threshold=0.80129395 positives=604,743 rate=0.18000 rows=3,359,679
submission_category_aware_blend_r20.csv: threshold=0.78042001 positives=671,936 rate=0.20000 rows=3,359,679
submission_category_aware_blend_r22.csv: threshold=0.76053247 positives=739,130 rate=0.22000 rows=3,359,679
submission_category_aware_blend_r24.csv: threshold=0.74172020 positives=806,325 rate=0.24000 rows=3,359,679


## Submission reminder

Do not submit more than the competition daily quota. On 2026-06-28 the quota was already used. When the quota resets, start with the blended 20 percent or 22 percent positive-rate files if the full run completes successfully.
